## PHASE 3
### Transform raw tables into useful business data

In [45]:
import pandas as pd

In [46]:
orders = pd.read_csv("../data/processed/orders_cleaned.csv",
    parse_dates=[
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ]
)

#parse_dates=[]
#It tells Pandas:
#"These columns contain dates. Convert them into Pandas datetime format when reading the file."
#Without parse_dates, Pandas may read them as strings (object).
#With parse_dates, Pandas converts it into a proper datetime value.

In [47]:
orders.dtypes

order_id                                    str
customer_id                                 str
order_status                                str
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
delivery_days                           float64
delivery_delay_days                     float64
dtype: object

In [48]:
order_items = pd.read_csv(
    "../data/processed/order_items_cleaned.csv",
    parse_dates=["shipping_limit_date"]
)

products = pd.read_csv(
    "../data/processed/products_cleaned.csv"
)

customers = pd.read_csv(
    "../data/processed/customers_cleaned.csv"
)

sellers = pd.read_csv(
    "../data/processed/sellers_cleaned.csv"
)

payments = pd.read_csv(
    "../data/processed/payments_cleaned.csv"
)

reviews = pd.read_csv(
    "../data/processed/reviews_cleaned.csv",
    parse_dates=[
        "review_creation_date",
        "review_answer_timestamp"
    ]
)

category_translation = pd.read_csv(
    "../data/processed/category_translation_cleaned.csv"
)

In [49]:
orders[["order_id", "customer_id", "order_status"]].head()

,order_id,customer_id,order_status
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered


In [50]:
order_items[
    ["order_id", "order_item_id", "product_id", "price"]
].head()

,order_id,order_item_id,product_id,price
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,58.90
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,239.90
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,199.00
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,12.99
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,199.90


In [51]:
order_item_data=orders.merge(
    order_items,
    on="order_id",
    how="left")

order_item_data.head()

#left join: Keep every order, and attach order-item information where it exists.

#keeps all orders even if it has no matching in order_item

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,delivery_days,delivery_delay_days,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,8.0,-8.0,1.0,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,2017-10-06 11:07:15,29.99,8.72
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,13.0,-6.0,1.0,595fac2a385ac33a80bd5114aec74eb8,289cdb325fb7e7f891c38608bf9e0962,2018-07-30 03:24:27,118.70,22.76
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,9.0,-18.0,1.0,aa4383b373c6aca5d8797843e5594415,4869f7a5dfa277a7dca6462dcf3b52b2,2018-08-13 08:55:23,159.90,19.22
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,13.0,-13.0,1.0,d0b61bfb1de832b15ba9d266ca96e5b0,66922902710d126a0e7d26b0e3805106,2017-11-23 19:45:59,45.00,27.20
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,2.0,-10.0,1.0,65266b2da20d04dbe00c5c2d3bb7859e,2c9e548be18521d1c43cde1c582c6de8,2018-02-19 20:31:37,19.90,8.72


In [52]:
order_item_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 113425 entries, 0 to 113424
Data columns (total 16 columns):
 #   Column                         Non-Null Count   Dtype         
---  ------                         --------------   -----         
 0   order_id                       113425 non-null  str           
 1   customer_id                    113425 non-null  str           
 2   order_status                   113425 non-null  str           
 3   order_purchase_timestamp       113425 non-null  datetime64[us]
 4   order_approved_at              113264 non-null  datetime64[us]
 5   order_delivered_carrier_date   111457 non-null  datetime64[us]
 6   order_delivered_customer_date  110196 non-null  datetime64[us]
 7   order_estimated_delivery_date  113425 non-null  datetime64[us]
 8   delivery_days                  110196 non-null  float64       
 9   delivery_delay_days            110196 non-null  float64       
 10  order_item_id                  112650 non-null  float64       
 11  product_id 

In [53]:
orders.shape

(99441, 10)

In [54]:
#This should be larger than the number of orders because one order can have multiple order items.
order_item_data.shape

(113425, 16)

This is where your earlier understanding of grain becomes extremely important.

Before:

orders
1 row ≈ 1 order

After:

order_item_data
1 row ≈ 1 order item

That means we've changed the grain.

## Merge product 

Suppose both DataFrames have a column city.
When you merge them, Pandas finds that city exists in both DataFrames.
So it needs different names for the two city columns.

suffixes=("", "_product")

The first suffix:""

means:
Don't add anything to the column coming from order_item_data.

The second suffix:"_product"

means:
Add _product to duplicate columns coming from products.

In [55]:
order_item_data= order_item_data.merge(
    products,
    on="product_id",
    how="left",
    suffixes=("","_product")
)
order_item_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 113425 entries, 0 to 113424
Data columns (total 24 columns):
 #   Column                         Non-Null Count   Dtype         
---  ------                         --------------   -----         
 0   order_id                       113425 non-null  str           
 1   customer_id                    113425 non-null  str           
 2   order_status                   113425 non-null  str           
 3   order_purchase_timestamp       113425 non-null  datetime64[us]
 4   order_approved_at              113264 non-null  datetime64[us]
 5   order_delivered_carrier_date   111457 non-null  datetime64[us]
 6   order_delivered_customer_date  110196 non-null  datetime64[us]
 7   order_estimated_delivery_date  113425 non-null  datetime64[us]
 8   delivery_days                  110196 non-null  float64       
 9   delivery_delay_days            110196 non-null  float64       
 10  order_item_id                  112650 non-null  float64       
 11  product_id 

In [56]:
order_item_data = order_item_data.merge(
    sellers,
    on="seller_id",
    how="left",
    suffixes=("", "_seller")
)
order_item_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 113425 entries, 0 to 113424
Data columns (total 27 columns):
 #   Column                         Non-Null Count   Dtype         
---  ------                         --------------   -----         
 0   order_id                       113425 non-null  str           
 1   customer_id                    113425 non-null  str           
 2   order_status                   113425 non-null  str           
 3   order_purchase_timestamp       113425 non-null  datetime64[us]
 4   order_approved_at              113264 non-null  datetime64[us]
 5   order_delivered_carrier_date   111457 non-null  datetime64[us]
 6   order_delivered_customer_date  110196 non-null  datetime64[us]
 7   order_estimated_delivery_date  113425 non-null  datetime64[us]
 8   delivery_days                  110196 non-null  float64       
 9   delivery_delay_days            110196 non-null  float64       
 10  order_item_id                  112650 non-null  float64       
 11  product_id 

In [58]:
order_item_data = order_item_data.merge(
    category_translation,
    on="product_category_name",
    how="left"
)

In [59]:
order_item_data.shape

(113425, 28)

In [60]:
order_item_data.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,delivery_days,delivery_delay_days,...,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,seller_zip_code_prefix,seller_city,seller_state,product_category_name_english
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,8.0,-8.0,...,268.0,4.0,500.0,19.0,8.0,13.0,9350.0,maua,SP,housewares
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,13.0,-6.0,...,178.0,1.0,400.0,19.0,13.0,19.0,31570.0,belo horizonte,SP,perfumery
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,9.0,-18.0,...,232.0,1.0,420.0,24.0,19.0,21.0,14840.0,guariba,SP,auto
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,13.0,-13.0,...,468.0,3.0,450.0,30.0,10.0,20.0,31842.0,belo horizonte,MG,pet_shop
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,2.0,-10.0,...,316.0,4.0,250.0,51.0,15.0,15.0,8752.0,mogi das cruzes,SP,stationery


In [63]:
order_item_data.columns.tolist()

['order_id',
 'customer_id',
 'order_status',
 'order_purchase_timestamp',
 'order_approved_at',
 'order_delivered_carrier_date',
 'order_delivered_customer_date',
 'order_estimated_delivery_date',
 'delivery_days',
 'delivery_delay_days',
 'order_item_id',
 'product_id',
 'seller_id',
 'shipping_limit_date',
 'price',
 'freight_value',
 'product_category_name',
 'product_name_lenght',
 'product_description_lenght',
 'product_photos_qty',
 'product_weight_g',
 'product_length_cm',
 'product_height_cm',
 'product_width_cm',
 'seller_zip_code_prefix',
 'seller_city',
 'seller_state',
 'product_category_name_english']

Check whether the joins worked

In [64]:
order_item_data["product_id"].isna().sum()

np.int64(775)

In [65]:
order_item_data["seller_id"].isna().sum()

np.int64(775)

In [66]:
order_item_data["product_category_name_english"].isna().sum()

np.int64(2402)

In [67]:
order_item_data["item_total"] = (
    order_item_data["price"]
    + order_item_data["freight_value"]
)

In [68]:
order_item_data[
    ["price", "freight_value", "item_total"]
].head()

,price,freight_value,item_total
0,29.99,8.72,38.71
1,118.70,22.76,141.46
2,159.90,19.22,179.12
3,45.00,27.20,72.20
4,19.90,8.72,28.62


In [74]:
#applying different aggregate function to different columns

order_summary=(order_item_data.groupby("order_id",as_index=False)).agg(
    order_item_revenue=("price","sum"),
    total_freight=("freight_value","sum"),
    total_order_value=("item_total",sum),
    item_count=("order_id","count")
)

order_summary.sort_values("item_count",ascending=False)


,order_id,order_item_revenue,total_freight,total_order_value,item_count
50543,8272b63d03f5f79c56e9e4120aec44ef,31.80,164.37,196.17,21
10541,1b15974a0141d54e36626dca3fdc731a,2000.00,202.40,2202.40,20
66248,ab14fdcfbe524636d65ee38360e22ce8,1974.00,288.80,2262.80,20
25797,428a2f660dc84138d969ccd69a0ab6d5,982.35,243.30,1225.65,15
61436,9ef13efd6949e4573a18964dd1bbe7f5,765.00,18.00,783.00,15
...,...,...,...,...,...
99420,fff1e3e76b816bfe8ef16678cc53c643,65.99,20.86,86.85,1
99421,fff2cdc825f9fc0ba3c04227cfa02303,24.99,25.63,50.62,1
99422,fff2e9e3aa8644e19710216b4ef53ab2,69.90,16.25,86.15,1
99423,fff3983dfa3c5a0d752d8d17baa406a0,66.39,14.05,80.44,1


In [75]:
order_summary.shape

(99441, 5)

In [76]:
order_summary["item_count"].describe()

count    99441.000000
mean         1.140626
std          0.536495
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max         21.000000
Name: item_count, dtype: float64

### Join customers

In [78]:
order_item_data=order_item_data.merge(
    customers,
    on="customer_id",
    how="left",
    suffixes=("","_customer")
)

In [86]:
payments.head()

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45


In [80]:
payments.columns.tolist()

['order_id',
 'payment_sequential',
 'payment_type',
 'payment_installments',
 'payment_value']

In [81]:
payments["payment_type"].value_counts()

payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64

In [84]:
payments.groupby("order_id").size().describe()

count    99440.000000
mean         1.044710
std          0.381166
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max         29.000000
dtype: float64

For example:

Order A
│
├── Credit card → ₹100
└── Voucher     → ₹50

becomes:

Order A
payment_value = ₹150
payment_count = 2

We have compressed multiple payment records into one order-level record.

In [87]:
payment_summary=(payments.groupby("order_id",as_index=False).agg(
    payment_value=("payment_value","sum"),
    payment_count=("payment_sequential","count"),
    payment_installments_max=("payment_installments","max")
))

payment_summary.head()

,order_id,payment_value,payment_count,payment_installments_max
0,00010242fe8c5a6d1ba2dd792cb16214,72.19,1,2
1,00018f77f2f0320c557190d7a144bdd3,259.83,1,3
2,000229ec398224ef6ca0657da4fc703e,216.87,1,5
3,00024acbcdf0a6daa1e931b038114c75,25.78,1,2
4,00042b26cf59d7ce69dfabb4e55b4fd9,218.04,1,3


In [89]:
order_item_data=order_item_data.merge(
    payment_summary,
    on="order_id",
    how="left"
)

Add Review Information

In [94]:
# size() counts the number of rows in each group.
# describe() gives statistical information about those counts.
reviews.groupby("order_id").size().describe()

count    98673.000000
mean         1.005584
std          0.075060
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max          3.000000
dtype: float64

In [95]:
reviews.columns.tolist()

['review_id',
 'order_id',
 'review_score',
 'review_comment_title',
 'review_comment_message',
 'review_creation_date',
 'review_answer_timestamp']

In [96]:
review_summary=(reviews.groupby("order_id",as_index=False).agg(
    review_score=("review_score","mean"),
    review_count=("review_id","nunique")
))

review_summary.head()

,order_id,review_score,review_count
0,00010242fe8c5a6d1ba2dd792cb16214,5.0,1
1,00018f77f2f0320c557190d7a144bdd3,4.0,1
2,000229ec398224ef6ca0657da4fc703e,5.0,1
3,00024acbcdf0a6daa1e931b038114c75,4.0,1
4,00042b26cf59d7ce69dfabb4e55b4fd9,5.0,1


In [97]:
order_item_data=order_item_data.merge(
    review_summary,
    on="order_id",
    how="left"
)

In [105]:
def classify_review(score):
    if pd.isna(score):
        return "No review"
    elif score<=2:
        return "poor"
    elif score == 3:
        return "average"
    else:
        return "good"

classify_review(2)

'poor'

In [107]:
order_item_data["review_category"]=(order_item_data["review_score"].apply(classify_review))
order_item_data["review_category"].value_counts()

review_category
good         84497
poor         18530
average       9437
No review      961
Name: count, dtype: int64

Add delivery performance

In [108]:
order_item_data[
    [
        "order_id",
        "delivery_days",
        "delivery_delay_days"
    ]
].head()

,order_id,delivery_days,delivery_delay_days
0,e481f51cbdc54678b7cc49136f2d6af7,8.0,-8.0
1,53cdb2fc8bc7dce0b6741e2150273451,13.0,-6.0
2,47770eb9100c2d0c44946d9cf07ec65d,9.0,-18.0
3,949d5b44dbf5de918fe9c16f97b45f8a,13.0,-13.0
4,ad21c59c0840e6cb83a9ceb5573f8159,2.0,-10.0


In [109]:
order_item_data["delivery_status"]="On Time"

In [111]:
order_item_data.loc[
    order_item_data["delivery_delay_days"]>0,
    "delivery_status"
]="Late"

In [112]:
order_item_data.loc[
    order_item_data["delivery_delay_days"].isna(),
    "delivery_status"
]="Not Delivered"

In [116]:
order_item_data["delivery_status"].value_counts()

delivery_status
On Time          102931
Late               7265
Not Delivered      3229
Name: count, dtype: int64

In [117]:
order_item_data.columns.tolist()

['order_id',
 'customer_id',
 'order_status',
 'order_purchase_timestamp',
 'order_approved_at',
 'order_delivered_carrier_date',
 'order_delivered_customer_date',
 'order_estimated_delivery_date',
 'delivery_days',
 'delivery_delay_days',
 'order_item_id',
 'product_id',
 'seller_id',
 'shipping_limit_date',
 'price',
 'freight_value',
 'product_category_name',
 'product_name_lenght',
 'product_description_lenght',
 'product_photos_qty',
 'product_weight_g',
 'product_length_cm',
 'product_height_cm',
 'product_width_cm',
 'seller_zip_code_prefix',
 'seller_city',
 'seller_state',
 'product_category_name_english',
 'item_total',
 'customer_unique_id',
 'customer_zip_code_prefix',
 'customer_city',
 'customer_state',
 'payment_value',
 'payment_count',
 'payment_installments_max',
 'review_score',
 'review_count',
 'review_category',
 'delivery_status']